# 🤖 LLM Agents with LangChain
### A Hands-On Workshop — 40 Minutes

---

**Platform:** Google Colab (no GPU needed — runs on free CPU)  
**Model:** OpenAI `gpt-4o-mini` via API — fast, cheap, and highly capable  
**Framework:** LangChain 0.3+

> 💰 **Cost estimate:** This entire workshop costs less than **$0.05** with `gpt-4o-mini`.

---

## 🗺️ Workshop Map

| Time | Section | What you'll learn |
|------|---------|-------------------|
| 0:00 | ⚙️ Setup | Install packages, configure API key |
| 0:05 | 📚 Section 1 — LLM Basics | Prompts, templates, chains |
| 0:15 | 🔧 Section 2 — Agents & Tools | ReAct pattern, tool use, AgentExecutor |
| 0:30 | 🧠 Section 3 — Memory | Stateful conversations, ConversationBufferMemory |
| 0:40 | 🎯 Wrap-Up | Key principles & next steps |

---

## 🎯 Learning Objectives

By the end of this workshop you will be able to:

1. Use **LangChain** to build prompt templates and basic LLM chains
2. Define **tools** and understand how agents decide when to use them
3. Implement the **ReAct** (Reason + Act) agent loop from scratch
4. Add **memory** to make agents aware of conversation history

> 💡 **Tip:** Run cells top to bottom. Each section builds on the previous one.

---
## ⚙️ Setup
*~5 minutes — run once at the start*

No GPU required. This notebook calls the OpenAI API over the network, so any Colab runtime works.

In [ ]:
# Install all required packages (~30 seconds)
!pip install -q \
    langchain==0.3.* \
    langchain-community==0.3.* \
    langchain-openai==0.2.* \
    openai

print("✅ Installation complete!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
langgraph 1.2.4 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is inco

### 🔑 API Key Configuration

You need an OpenAI API key from [platform.openai.com](https://platform.openai.com/api-keys).

**Two ways to set it — choose one:**

**Option A (recommended) — Colab Secrets:**  
Click the 🔑 key icon in the left sidebar → "Add new secret" → name it `OPENAI_API_KEY` → paste your key. Then run the cell below and flip the "Notebook access" toggle.

**Option B — Direct input:**  
Uncomment the `getpass` line in the cell below. You'll be prompted to paste your key (it won't be shown on screen).

> ⚠️ Never paste your API key directly into a code cell — it gets saved in the notebook file.

In [ ]:
import os

# ── Option A: Colab Secrets (recommended) ───────────────────────────────────
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("✅ API key loaded from Colab Secrets")
except Exception:
    print("⚠️  Colab Secrets not found — falling back to manual input")

    # ── Option B: Manual input (uncomment if Option A fails) ────────────────
    from getpass import getpass
    os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")
    print("✅ API key set")

⚠️  Colab Secrets not found — falling back to manual input
Paste your OpenAI API key: ··········
✅ API key set


In [ ]:
import warnings
warnings.filterwarnings("ignore")

# LangChain core
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate, ChatPromptTemplate
from langchain.chains import LLMChain, ConversationChain
from langchain.memory import ConversationBufferMemory
from langchain.tools import tool

# ── Configure the model ─────────────────────────────────────────────────────
# MODEL OPTIONS (swap anytime):
#   "gpt-4o-mini"   — default: cheap, fast, very capable  (~$0.15/M input tokens)
#   "gpt-4o"        — most capable, higher cost           (~$2.50/M input tokens)
#   "gpt-3.5-turbo" — older, cheapest option              (~$0.50/M input tokens)

MODEL_NAME = "gpt-4o-mini"   # ← change this to experiment

llm = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0,      # 0 = deterministic, good for demos and exercises
)

# Quick connectivity check
test = llm.invoke("Reply with exactly: OK")
print(f"✅ Connected to {MODEL_NAME} — response: {test.content}")

✅ Connected to gpt-4o-mini — response: OK


---
## 📚 Section 1 — LLM Basics
*~10 minutes*

### Key Concept: What is a Language Model?

A **Large Language Model (LLM)** is a function that takes text as input and produces text as output:

```
Input Prompt  ──►  [ LLM ]  ──►  Generated Text
```

**LangChain** is a framework that makes it easy to:
- Build reusable **prompt templates**
- Chain multiple LLM calls together
- Connect LLMs to tools, databases, and memory

### 1.1 — Calling the LLM directly

In [ ]:
# The simplest possible interaction: pass a string, get a string back
# With ChatOpenAI, the response is a message object — use .content to get the text
response = llm.invoke("What is the capital of France?")
print(f"Response: {response.content}")
print()
print(f"Full response object type: {type(response)}")
print(f"Model used: {response.response_metadata.get('model_name', MODEL_NAME)}")

Response: The capital of France is Paris.

Full response object type: <class 'langchain_core.messages.ai.AIMessage'>
Model used: gpt-4o-mini-2024-07-18


### 1.2 — Prompt Templates

Hardcoding prompts is fragile. **PromptTemplate** lets you define a reusable prompt with **variables** (like `{topic}` or `{audience}`) that get filled in at runtime.

```
Template:  "Explain {topic} in simple terms for a {audience}."
                  ↓ fill variables
Prompt:    "Explain machine learning in simple terms for a beginner."
```

With chat models, LangChain also supports **ChatPromptTemplate**, which lets you assign roles (`system`, `human`, `ai`) to each message — great for controlling model behavior.

In [ ]:
# ── Basic PromptTemplate ─────────────────────────────────────────────────────
explain_template = PromptTemplate(
    input_variables=["topic", "audience"],
    template="Explain {topic} in simple terms for a {audience}. Be brief (2-3 sentences)."
)

# Preview what the filled prompt looks like before sending to the LLM
filled_prompt = explain_template.format(topic="neural networks", audience="10-year-old")
print("Filled prompt:")
print(filled_prompt)
print()

# Send it to the LLM
response = llm.invoke(filled_prompt)
print("LLM response:")
print(response.content)

Filled prompt:
Explain neural networks in simple terms for a 10-year-old. Be brief (2-3 sentences).

LLM response:
Neural networks are like a group of tiny helpers in a computer that work together to solve problems, kind of like how our brains work. They learn from examples, like how you learn to recognize animals by looking at pictures, and then they can help the computer make decisions or predictions based on what they've learned.


In [ ]:
# ── ChatPromptTemplate — gives you system + user message separation ──────────
# The 'system' message sets the persona/behavior of the model.
# The 'human' message is what the user says.

chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a {persona}. Always respond in a tone consistent with that persona."),
    ("human",  "{question}"),
])

# Test with different personas
examples = [
    {"persona": "pirate",             "question": "What is machine learning?"},
    {"persona": "formal academic",    "question": "What is machine learning?"},
    {"persona": "5-year-old child",   "question": "What is machine learning?"},
]

for ex in examples:
    messages = chat_template.format_messages(**ex)
    response = llm.invoke(messages)
    print(f"🎭 Persona: {ex['persona']}")
    print(f"   {response.content}")
    print("-" * 60)

🎭 Persona: pirate
   Arrr, matey! Machine learnin' be a fine treasure of the tech seas! It be a branch of computer science where scallywags be teachin' machines to learn from data, just like a young buccaneer learnin' the ropes of the high seas. Instead of bein' programmed with every command, these clever contraptions use algorithms to find patterns and make decisions on their own, like a captain navigatin' by the stars!

Ye see, it be all about givin' the machines a heap o' data, and they be usin' that to improve their performance over time, just like a crew gettin' better at sailin' with practice. From predictin' the weather to recommendin' the best rum, machine learnin' be a powerful tool in the modern world, aye! So hoist the sails and let the data flow, for the journey of learnin' be never-ending!
------------------------------------------------------------
🎭 Persona: formal academic
   Machine learning is a subfield of artificial intelligence (AI) that focuses on the development 

### 1.3 — LLM Chains

An **LLMChain** bundles a prompt template + an LLM into a single reusable unit. You pass the variables, it returns the output.

```
Variables ──► PromptTemplate ──► LLM ──► Output
      └──────────────────────────────────┘
                    LLMChain
```

In [ ]:
# Build the chain — template + LLM bundled together
explain_chain = LLMChain(llm=llm, prompt=explain_template)

# Call it for different topics — reuse the chain, just swap variables
topics = [
    {"topic": "photosynthesis",    "audience": "beginner"},
    {"topic": "blockchain",        "audience": "non-technical manager"},
    {"topic": "gradient descent",  "audience": "high school student"},
]

for item in topics:
    result = explain_chain.invoke(item)
    print(f"Topic: {item['topic']}")
    print(f"Response: {result['text']}")
    print("-" * 60)

/tmp/ipykernel_6361/1116767069.py:2: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  explain_chain = LLMChain(llm=llm, prompt=explain_template)


Topic: photosynthesis
Response: Photosynthesis is the process by which plants, algae, and some bacteria convert sunlight into energy. They use sunlight, carbon dioxide from the air, and water from the soil to produce glucose (a type of sugar) and oxygen. This process is essential for life on Earth, as it provides food for plants and oxygen for animals and humans.
------------------------------------------------------------
Topic: blockchain
Response: Blockchain is a secure and transparent digital ledger that records transactions across multiple computers, ensuring that the information is tamper-proof and easily verifiable. Each transaction is grouped into a "block," and once a block is filled, it is linked to the previous one, forming a "chain." This technology enhances trust and accountability in various processes by providing a clear and immutable record of all activities.
------------------------------------------------------------
Topic: gradient descent
Response: Gradient descent 

### ✏️ Exercise 1 — Build Your Own Chain

Create a chain that takes a `programming_language` variable and generates a **one-sentence description** of what it's best used for.

Expected output (example):
```
Python → Python is best used for data science, AI, and scripting.
Rust   → Rust is best used for systems programming where speed and safety matter.
```

Fill in the `???` below:

In [ ]:
# YOUR CODE HERE
lang_template = PromptTemplate(
    input_variables=["programming_language"],
    template=???  # Write your prompt here
)

lang_chain = LLMChain(llm=llm, prompt=lang_template)

for lang in ["Python", "Rust", "JavaScript"]:
    result = lang_chain.invoke({"programming_language": lang})
    print(f"{lang} → {result['text']}")

In [ ]:
# ✅ Exercise 1 — Solution (run only after trying yourself!)
lang_template = PromptTemplate(
    input_variables=["programming_language"],
    template="In one sentence, what is {programming_language} best used for?"
)
lang_chain = LLMChain(llm=llm, prompt=lang_template)

for lang in ["Python", "Rust", "JavaScript"]:
    result = lang_chain.invoke({"programming_language": lang})
    print(f"{lang} → {result['text']}")

---
## 🔧 Section 2 — Agents & Tool Use
*~15 minutes*

### Key Concept: What is an Agent?

A plain LLM chain always runs the same fixed steps. An **agent** is different — it uses the LLM to **decide what to do next**.

```
Question
   │
   ▼
[ LLM thinks ]
   │
   ├──► "I need to use a tool"  ──►  [ Run Tool ]  ──► Observation ──┐
   │                                                                   │
   │◄──────────────────────────────────────────────────────────────────┘
   │
   └──► "I have the answer"  ──►  Final Answer
```

### The ReAct Pattern

**ReAct** = **Re**ason + **Act** — a framework where the LLM alternates between:

| Step | What happens |
|------|--------------|
| **Thought** | The LLM reasons about what it knows and what it needs |
| **Action** | The LLM picks a tool and an input for it |
| **Observation** | The tool runs and returns a result |
| *(repeat)* | Until the LLM has enough information |
| **Final Answer** | The LLM produces the final response |

> 🔍 **Why does this matter?** LLMs are great at *reasoning* but bad at *precision tasks* (exact math, live data, code execution). Tools bridge that gap — the LLM decides *when* and *how* to use them.

---

### 2.1 — Defining Tools

A **tool** is just a Python function decorated with `@tool`. LangChain reads the **function name** and **docstring** to know what the tool does — the docstring is literally the description sent to the model.

In [ ]:
import math

# ── Tool 1: Calculator ───────────────────────────────────────────────────────
@tool
def calculator(expression: str) -> str:
    """Evaluates a mathematical expression. Input must be a valid Python math
    expression string. Examples: '2 + 2', '15 * 8', 'math.sqrt(144)', '2 ** 10'"""
    try:
        result = eval(expression, {"__builtins__": {}}, {"math": math})
        return str(result)
    except Exception as e:
        return f"Error: {e}"


# ── Tool 2: Country Capital Lookup ──────────────────────────────────────────
CAPITALS = {
    "france": "Paris", "germany": "Berlin", "japan": "Tokyo",
    "brazil": "Brasília", "australia": "Canberra", "egypt": "Cairo",
    "canada": "Ottawa", "india": "New Delhi", "mexico": "Mexico City",
    "argentina": "Buenos Aires", "south korea": "Seoul", "spain": "Madrid",
    "italy": "Rome", "portugal": "Lisbon", "netherlands": "Amsterdam",
}

@tool
def capital_lookup(country: str) -> str:
    """Looks up the capital city of a country. Input: country name (lowercase).
    Examples: 'france', 'japan', 'brazil'"""
    key = country.strip().lower()
    return CAPITALS.get(key, f"Capital of '{country}' not found in database.")


# ── Tool 3: Word Counter ─────────────────────────────────────────────────────
@tool
def word_counter(text: str) -> str:
    """Counts the number of words in a text string. Input: any string of text."""
    count = len(text.split())
    return f"{count} words"


# Bundle into a list
TOOLS = [calculator, capital_lookup, word_counter]

# ── Inspect what the tools look like to the model ───────────────────────────
print("Available tools:")
for t in TOOLS:
    print(f"  • {t.name}")
    print(f"    {t.description[:100]}")
    print()

# Test each tool directly (without the LLM)
print("Direct tool calls:")
print(f"  calculator('2 ** 10')        → {calculator.invoke('2 ** 10')}")
print(f"  capital_lookup('japan')      → {capital_lookup.invoke('japan')}")
print(f"  word_counter('hello world')  → {word_counter.invoke('hello world')}")

Available tools:
  • calculator
    Evaluates a mathematical expression. Input must be a valid Python math
    expression string. Exampl

  • capital_lookup
    Looks up the capital city of a country. Input: country name (lowercase).
    Examples: 'france', 'ja

  • word_counter
    Counts the number of words in a text string. Input: any string of text.

Direct tool calls:
  calculator('2 ** 10')        → 1024
  capital_lookup('japan')      → Tokyo
  word_counter('hello world')  → 2 words


### 2.2 — The ReAct Loop (from scratch)

Before using LangChain's agent abstraction, let's **build the loop manually**. This makes the mechanics transparent.

The algorithm is:

```python
while not done:
    thought = llm.think(question + history)   # Reason
    action  = llm.pick_tool(thought)          # Decide
    obs     = tools[action].run(input)        # Act
    history.append(thought + action + obs)    # Observe

return llm.final_answer(history)
```

With a capable model like `gpt-4o-mini`, we can ask it to output the full Thought / Action / Action Input format that the ReAct paper describes, and parse it reliably.

In [ ]:
import re

REACT_SYSTEM_PROMPT = """\
You are a helpful assistant that solves problems step by step using available tools.

Available tools:
{tool_descriptions}

To use a tool, respond EXACTLY in this format:
Thought: <your reasoning about what to do next>
Action: <tool name>
Action Input: <exact input to pass to the tool>

When you have the final answer (no more tools needed), respond EXACTLY in this format:
Thought: <your reasoning>
Final Answer: <your complete answer>

Never skip the Thought step."""


def parse_react_output(text):
    """Parse the model's ReAct-format response into structured parts."""
    thought  = re.search(r"Thought:\s*(.+?)(?=\n(?:Action|Final Answer)|$)", text, re.DOTALL)
    action   = re.search(r"Action:\s*(.+)", text)
    action_i = re.search(r"Action Input:\s*(.+)", text)
    final    = re.search(r"Final Answer:\s*(.+)", text, re.DOTALL)

    return {
        "thought":      thought.group(1).strip()  if thought  else "",
        "action":       action.group(1).strip()   if action   else None,
        "action_input": action_i.group(1).strip() if action_i else None,
        "final_answer": final.group(1).strip()    if final    else None,
    }


def manual_react_agent(question, tools, llm, max_steps=5, verbose=True):
    """A ReAct agent implemented from scratch — no LangChain agent magic."""
    tool_map = {t.name: t for t in tools}

    # Build the tool descriptions for the system prompt
    tool_descriptions = "\n".join(
        [f"- {t.name}: {t.description.split(chr(10))[0]}" for t in tools]
    )

    # Conversation history (system message + running turns)
    messages = [
        {"role": "system",  "content": REACT_SYSTEM_PROMPT.format(tool_descriptions=tool_descriptions)},
        {"role": "user",    "content": question},
    ]

    if verbose:
        print(f"❓ Question: {question}")
        print("=" * 60)

    for step in range(1, max_steps + 1):
        # ── Ask the model what to do next ────────────────────────────────────
        response = llm.invoke(messages)
        raw_text = response.content
        parsed   = parse_react_output(raw_text)

        if verbose:
            print(f"\n🧠 Step {step}")
            print(f"   Thought: {parsed['thought']}")

        # ── Final answer reached ──────────────────────────────────────────────
        if parsed["final_answer"]:
            if verbose:
                print(f"\n✅ Final Answer: {parsed['final_answer']}")
            return parsed["final_answer"]

        # ── Run the chosen tool ───────────────────────────────────────────────
        action = parsed["action"]
        action_input = parsed["action_input"] or ""

        if action not in tool_map:
            observation = f"Tool '{action}' not found. Available: {list(tool_map.keys())}"
        else:
            observation = tool_map[action].invoke(action_input)

        if verbose:
            print(f"   Action: {action}({repr(action_input)})")
            print(f"   Observation: {observation}")

        # Append to conversation history so the model sees what happened
        messages.append({"role": "assistant", "content": raw_text})
        messages.append({"role": "user",      "content": f"Observation: {observation}"})

    return "Max steps reached without a final answer."

In [ ]:
# ── Watch the Thought → Action → Observation loop step by step ───────────────
print("=" * 60)
manual_react_agent(
    question="What is 15 multiplied by 23?",
    tools=TOOLS, llm=llm, verbose=True
)

print("\n" + "=" * 60)
manual_react_agent(
    question="What is the capital of Brazil and how many letters does it have?",
    tools=TOOLS, llm=llm, verbose=True
)

❓ Question: What is 15 multiplied by 23?

🧠 Step 1
   Thought: I need to calculate the product of 15 and 23.
   Action: calculator('15 * 23')
   Observation: 345

🧠 Step 2
   Thought: I have the result of the multiplication, which is 345. Now I will present the final answer.

✅ Final Answer: 15 multiplied by 23 is 345.

❓ Question: What is the capital of Brazil and how many letters does it have?

🧠 Step 1
   Thought: First, I need to find out the capital of Brazil. After that, I will count the number of letters in the name of the capital city.
   Action: capital_lookup('brazil')
   Observation: Brasília

🧠 Step 2
   Thought: The capital of Brazil is Brasília. Now, I will count the number of letters in the name "Brasília."
   Action: word_counter('Brasília')
   Observation: 1 words

🧠 Step 3
   Thought: The name "Brasília" consists of 1 word, but I need to count the number of letters in it. "Brasília" has 8 letters.

✅ Final Answer: The capital of Brazil is Brasília, which has 8 letters

'The capital of Brazil is Brasília, which has 8 letters.'

### 2.3 — LangChain's AgentExecutor

LangChain automates what we just built with `AgentExecutor`. It handles the loop, output parsing, retries, and error recovery — using the exact same ReAct logic under the hood.

Notice how `verbose=True` prints a trace that matches our manual Thought/Action/Observation format.

In [ ]:
from langchain.agents import AgentExecutor, create_react_agent
from langchain.prompts import PromptTemplate

# Define the standard ReAct prompt inline.
# This is the same content as hwchase17/react on LangChain Hub — defining it
# here avoids Hub authentication issues and lets you see exactly what the
# model receives on every step.
REACT_TEMPLATE = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

react_prompt = PromptTemplate(
    input_variables=["tools", "tool_names", "input", "agent_scratchpad"],
    template=REACT_TEMPLATE,
)

print("ReAct prompt template (first 500 chars):")
print(react_prompt.template[:500], "...\n")

ReAct prompt template (first 500 chars):
Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the ...



In [ ]:
# Build the agent
agent = create_react_agent(
    llm=llm,
    tools=TOOLS,
    prompt=react_prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=TOOLS,
    verbose=True,               # prints the full Thought/Action/Observation trace
    handle_parsing_errors=True,
    max_iterations=6
)

# Run it — compare this trace to our manual loop!
result = agent_executor.invoke({"input": "What is the square root of 256?"})
print(f"\n🎯 Answer: {result['output']}")



> Entering new AgentExecutor chain...
To find the square root of 256, I can use the calculator tool to evaluate the expression `math.sqrt(256)`.  
Action: calculator  
Action Input: 'math.sqrt(256)'  math.sqrt(256)The square root of 256 is 16.  
Final Answer: 16

> Finished chain.

🎯 Answer: 16


In [ ]:
# Test a question that requires two tool calls
result = agent_executor.invoke({
    "input": "What is the capital of Italy, and how many letters does that city name have?"
})
print(f"\n🎯 Answer: {result['output']}")



> Entering new AgentExecutor chain...
I need to find the capital of Italy first, and then count the number of letters in that city name. 

Action: capital_lookup  
Action Input: 'italy'  Capital of ''italy'' not found in database.It seems there was an issue with the lookup for Italy's capital. I will try again to find the capital of Italy.

Action: capital_lookup  
Action Input: 'italy'  Capital of ''italy'' not found in database.It seems that the lookup for Italy's capital is not working. However, I know that the capital of Italy is Rome. Now, I will count the number of letters in "Rome."

Action: word_counter  
Action Input: 'Rome'  1 wordsI need to count the number of letters in the word "Rome," not the number of words. 

Action: calculator  
Action Input: 'len("Rome")'  len("Rome")I need to evaluate the length of the string "Rome" to find out how many letters it has.

Action: calculator  
Action Input: '4'  4I now know the final answer.  
Final Answer: The capital of Italy is Rom

### ✏️ Exercise 2 — Add a New Tool

The agents currently have: `calculator`, `capital_lookup`, `word_counter`.

**Your task:** Add a `temperature_converter` tool that converts Celsius to Fahrenheit.

Formula: `F = (C × 9/5) + 32`

Then test the agent with: *"Convert 100 degrees Celsius to Fahrenheit, then calculate half of that value."*  
*(This requires two tool calls: converter → calculator)*

In [ ]:
# YOUR CODE HERE
@tool
def temperature_converter(celsius: str) -> str:
    """Converts a temperature from Celsius to Fahrenheit. Input: temperature in Celsius as a number string."""
    try:
        c = float(celsius)
        f = ???  # Your formula here
        return f"{c}°C = {f:.1f}°F"
    except Exception as e:
        return f"Error: {e}"

# Build a new executor with the extended tool list
MY_TOOLS = TOOLS + [temperature_converter]
my_agent = create_react_agent(llm=llm, tools=MY_TOOLS, prompt=react_prompt)
my_executor = AgentExecutor(agent=my_agent, tools=MY_TOOLS, verbose=True,
                             handle_parsing_errors=True, max_iterations=6)

result = my_executor.invoke({
    "input": "Convert 100 degrees Celsius to Fahrenheit, then calculate half of that value."
})
print(f"\n🎯 Answer: {result['output']}")

In [ ]:
# ✅ Exercise 2 — Solution
@tool
def temperature_converter(celsius: str) -> str:
    """Converts a temperature from Celsius to Fahrenheit. Input: temperature in Celsius as a number string."""
    try:
        c = float(celsius)
        f = (c * 9 / 5) + 32
        return f"{c}°C = {f:.1f}°F"
    except Exception as e:
        return f"Error: {e}"

MY_TOOLS = TOOLS + [temperature_converter]
my_agent = create_react_agent(llm=llm, tools=MY_TOOLS, prompt=react_prompt)
my_executor = AgentExecutor(agent=my_agent, tools=MY_TOOLS, verbose=True,
                             handle_parsing_errors=True, max_iterations=6)

result = my_executor.invoke({
    "input": "Convert 100 degrees Celsius to Fahrenheit, then calculate half of that value."
})
print(f"\n🎯 Answer: {result['output']}")

---
## 🧠 Section 3 — Memory & State
*~10 minutes*

### The Problem: LLMs are Stateless

Every API call is **independent** — the model has no memory of previous messages:

```
Turn 1: "My name is Alice."     → "Nice to meet you, Alice!"
Turn 2: "What is my name?"      → "I don't know your name."  ← ❌ forgot!
```

The fix is simple: **include the conversation history in every new request**.

```
Turn 2 (with memory): "[Previous: Human: My name is Alice. AI: Nice to meet you!]\nWhat is my name?"
                       → "Your name is Alice."  ← ✅
```

**Memory** objects handle this automatically — they accumulate the history and inject it into the prompt.

### Memory Types

| Type | What it stores | Best for |
|------|---------------|----------|
| `ConversationBufferMemory` | **Full** history verbatim | Short conversations |
| `ConversationSummaryMemory` | LLM-generated **summary** | Long conversations |
| `ConversationBufferWindowMemory` | Last **N** turns only | Fixed-length context |

We'll use `ConversationBufferMemory` — the simplest and most transparent.

---

### 3.1 — The Problem Illustrated

In [ ]:
# Without memory: each call is completely isolated
print("--- Without Memory ---")

r1 = llm.invoke("My favorite color is blue. Please remember this.")
print(f"Turn 1: {r1.content}")

r2 = llm.invoke("What is my favorite color?")
print(f"Turn 2: {r2.content}")   # Will NOT know — each .invoke() is a fresh call

--- Without Memory ---
Turn 1: I can't store personal information or remember details between our conversations. However, I can help you with anything related to the color blue or any other topic you'd like to discuss!
Turn 2: I don't have access to personal information about you, so I can't know your favorite color. However, if you tell me what it is, I'd be happy to chat about it!


### 3.2 — Adding Memory with ConversationChain

In [ ]:
# ConversationBufferMemory keeps the full conversation history.
# ConversationChain automatically prepends that history to every new message.

memory = ConversationBufferMemory()

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=False   # set True to see the full prompt being built each turn
)

print("Default prompt template:")
print(conversation.prompt.template)

Default prompt template:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
{history}
Human: {input}
AI:


/tmp/ipykernel_6361/258378484.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()
/tmp/ipykernel_6361/258378484.py:6: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use :class:`~langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  conversation = ConversationChain(


In [ ]:
# ── Multi-turn conversation — the model remembers across turns ───────────────
turns = [
    "My name is Alice and I'm learning about AI agents.",
    "What is my name?",
    "What topic am I learning about?",
    "Summarize what you know about me in one sentence.",
]

print("--- With Memory ---")
for i, turn in enumerate(turns, 1):
    response = conversation.predict(input=turn)
    print(f"\n[Turn {i}] Human: {turn}")
    print(f"         AI:    {response}")

--- With Memory ---

[Turn 1] Human: My name is Alice and I'm learning about AI agents.
         AI:    Hi Alice! It's great to meet you! AI agents are fascinating. They can perform tasks, make decisions, and even learn from their experiences. Are you focusing on a specific type of AI agent, like chatbots, virtual assistants, or something else? I’d love to hear more about what you’re learning!

[Turn 2] Human: What is my name?
         AI:    Your name is Alice! It's nice to meet you, Alice. How's your learning about AI agents going?

[Turn 3] Human: What topic am I learning about?
         AI:    You're learning about AI agents! That's an exciting topic with so many different aspects to explore. Are you looking into their applications, how they work, or perhaps the ethical considerations surrounding them? Let me know what interests you the most!

[Turn 4] Human: Summarize what you know about me in one sentence.
         AI:    You're Alice, and you're currently learning about AI agent

In [ ]:
# Inspect the raw memory buffer — this is exactly what gets injected into the prompt
print("=" * 60)
print("📦 Memory buffer contents:")
print("=" * 60)
print(memory.buffer)

📦 Memory buffer contents:
Human: My name is Alice and I'm learning about AI agents.
AI: Hi Alice! It's great to meet you! AI agents are fascinating. They can perform tasks, make decisions, and even learn from their experiences. Are you focusing on a specific type of AI agent, like chatbots, virtual assistants, or something else? I’d love to hear more about what you’re learning!
Human: What is my name?
AI: Your name is Alice! It's nice to meet you, Alice. How's your learning about AI agents going?
Human: What topic am I learning about?
AI: You're learning about AI agents! That's an exciting topic with so many different aspects to explore. Are you looking into their applications, how they work, or perhaps the ethical considerations surrounding them? Let me know what interests you the most!
Human: Summarize what you know about me in one sentence.
AI: You're Alice, and you're currently learning about AI agents, which is an exciting and multifaceted topic!


### 3.3 — Memory + Agents

Memory and tool use can be combined. We pass a `memory` object to `AgentExecutor` — it handles injecting history into each agent step automatically.

In [ ]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain.prompts import PromptTemplate

# For a memory-enabled agent we need a prompt that includes {chat_history}
MEMORY_REACT_PROMPT = PromptTemplate.from_template("""\
You are a helpful assistant with access to tools. Answer questions step by step.

Tools available:
{tools}

Tool names: {tool_names}

Previous conversation:
{chat_history}

Use this format:
Thought: your reasoning
Action: tool name
Action Input: tool input
Observation: tool result
... (repeat as needed)
Thought: I have the final answer
Final Answer: your answer

Question: {input}
{agent_scratchpad}""")

# Create the agent with memory
agent_memory = ConversationBufferMemory(
    memory_key="chat_history",   # must match the {chat_history} variable above
    return_messages=False
)

memory_agent = create_react_agent(llm=llm, tools=TOOLS, prompt=MEMORY_REACT_PROMPT)

memory_executor = AgentExecutor(
    agent=memory_agent,
    tools=TOOLS,
    memory=agent_memory,
    verbose=False,              # set True to see the full trace
    handle_parsing_errors=True,
    max_iterations=6
)

print("✅ Memory-enabled agent ready!")

✅ Memory-enabled agent ready!


In [ ]:
# Multi-turn conversation — the agent uses tools AND remembers previous turns
queries = [
    "What is the capital of Japan?",
    "What is 7 times 8?",
    "What were the two things I just asked you?",
]

print("--- Agent with Memory + Tools ---")
for q in queries:
    result = memory_executor.invoke({"input": q})
    print(f"\n👤 Human: {q}")
    print(f"🤖 Agent: {result['output']}")

--- Agent with Memory + Tools ---

👤 Human: What is the capital of Japan?
🤖 Agent: Tokyo

👤 Human: What is 7 times 8?
🤖 Agent: Agent stopped due to iteration limit or time limit.

👤 Human: What were the two things I just asked you?
🤖 Agent: The capital of Japan is Tokyo, and 7 times 8 is 56.


### ✏️ Exercise 3 — Context-Aware Follow-Ups

Continue the `memory_executor` conversation above with follow-up questions that test whether memory is working.

Try things like:
- `"What country did I ask about earlier?"`
- `"Double the number from my multiplication question."`
- `"How many letters are in the capital city I asked about?"`

In [ ]:
# YOUR CODE HERE — the memory_executor is still active from above
follow_up = "???"  # Replace with your question

result = memory_executor.invoke({"input": follow_up})
print(f"👤 Human: {follow_up}")
print(f"🤖 Agent: {result['output']}")

👤 Human: ???
🤖 Agent: The capital of Japan is Tokyo, and 7 times 8 is 56.


In [ ]:
# ✅ Exercise 3 — Example Solution
follow_ups = [
    "What country did I ask about earlier?",
    "Double the number from my multiplication question.",
    "How many letters are in the capital city I asked about?",
]

for q in follow_ups:
    result = memory_executor.invoke({"input": q})
    print(f"👤 Human: {q}")
    print(f"🤖 Agent: {result['output']}")
    print()

---
## 🎯 Wrap-Up

### Core Principles Covered

| Principle | What you learned |
|-----------|------------------|
| **Prompt Templates** | Reusable, parameterized prompts — no string concatenation |
| **Chat Templates** | System / Human / AI message roles for behavior control |
| **LLM Chains** | Compose prompt + LLM into a single callable unit |
| **Tools** | Give LLMs access to deterministic, precise capabilities |
| **ReAct Loop** | Agents reason and act iteratively until they reach an answer |
| **Memory** | Inject history into each prompt to make agents stateful |

---

### The Big Picture

```
               ┌────────────────────────────────────────────────────┐
               │                    LLM AGENT                        │
               │                                                      │
  User Input ──► Memory (context) + LLM (reasoning) ──► Tools ──► Answer
               │                                                      │
               └────────────────────────────────────────────────────┘
```

---

### Next Steps

**Upgrade the model** — swap `MODEL_NAME` at the top to experiment:
```python
MODEL_NAME = "gpt-4o"       # most capable
MODEL_NAME = "gpt-3.5-turbo" # cheapest
```

**Add real tools** — connect to live data:
```python
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun
search = DuckDuckGoSearchRun()   # no API key needed
```

**Add RAG** — ground agents in your own documents:
```python
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
# load docs → embed → retrieve → answer
```

**LangGraph** — for complex workflows with branching, cycles, and human-in-the-loop approval.

---

### Useful Resources

- [LangChain Docs](https://python.langchain.com/docs/get_started/introduction)
- [OpenAI Platform](https://platform.openai.com)
- [ReAct Paper](https://arxiv.org/abs/2210.03629)
- [LangGraph](https://langchain-ai.github.io/langgraph/)
- [LangChain Hub](https://smith.langchain.com/hub) — community prompts

---

*Happy building! 🚀*